# Week 6 · Day 1 — Your first SQL, run the Snowflake way

*Stop bringing the data to your laptop. Send the question to the data instead.*

**By the end you'll have shipped:** a handful of real **SQL queries** — `SELECT` / `WHERE` / `ORDER BY` / `LIMIT` — run against a `coffee_orders` table through **one reusable connection that talks to Snowflake in production and to a local engine (DuckDB) offline**. Same SQL, no account required.

### 📋 Lesson card

| | |
|---|---|
| **Module** | M1b · SQL foundations → M4 · Snowflake (Week 6) |
| **Prerequisites** | Week 2 (esp. Day 1 — pandas `select / filter / sort / groupby`) |
| **Est. time** | ~30 min |
| **Capstone slice** | *Store & query* legal matters — the on-ramp to keeping *Matter Intelligence* data in Snowflake |
| **Difficulty** | Core (everyone) + `Go Deeper 🔧` |
| **Runs offline?** | ✅ Yes — no Snowflake account, no key, no internet (falls back to DuckDB) |

### 🎯 Learning objectives

By the end you'll be able to:
- Explain what a **data warehouse** is and why the firm's matters will live in one.
- Connect to a warehouse with **one reusable helper** — Snowflake when credentials exist, **DuckDB** offline — writing the *same SQL* either way.
- Write **`SELECT`** (pick columns), **`WHERE`** (filter rows), **`ORDER BY`** (sort), and **`LIMIT`** (cap results).
- Pull query results **straight into a pandas DataFrame** for the Python half of your work.
- Map every SQL verb to the **pandas move you already know**.

### ⚖️ Why it matters

In Weeks 1–2 you loaded a CSV into pandas with `pd.read_csv`. That's perfect — right up until the table is **50 million rows** and lives on a server the whole firm shares. You can't download the firm's entire matter history to your laptop every morning.

**SQL** is how you ask questions of data *where it lives*. You write a question ("the ten biggest active matters"), send it to the warehouse, and get back only the answer. **Snowflake** is the warehouse the firm will use — and its query language is SQL. So learning SQL *is* learning to use Snowflake.

> **The plan for Weeks 6–7:** learn the SQL verbs on a relatable **coffee-orders** table (this week), then apply each one to **matters** — and in Week 7 you'll create tables, load data, and even run an LLM *inside* Snowflake with Cortex.

### ⚙️ Setup — one connection, two backends

Run this once. It builds a single helper, **`run_sql(...)`**, that returns a pandas DataFrame.

- If your `.env` has `SNOWFLAKE_*` credentials → it runs your SQL **on Snowflake**.
- If not (the default today) → it runs the **exact same SQL on DuckDB**, a fast local engine whose dialect is very close to Snowflake's. Nothing to sign up for.

Either way, **you write real Snowflake SQL.** The backend is invisible.

> 🔒 *Synthetic data only — this coffee set (and the matters set) is fake. Never load real client or privileged data into a teaching notebook.*

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")
import pandas as pd

# Load .env if python-dotenv is installed (optional — the notebook runs without it).
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# --- Decide the backend: Snowflake if credentials exist, else local DuckDB ---
SNOWFLAKE_READY = all(os.environ.get(k) for k in ("SNOWFLAKE_ACCOUNT", "SNOWFLAKE_USER", "SNOWFLAKE_PASSWORD"))
BACKEND = "snowflake" if SNOWFLAKE_READY else "duckdb"

# --- Find the shared coffee data (or fall back to a tiny built-in sample) ---
CSV_PATH = None
for p in ("../../data/coffee_orders.csv", "data/coffee_orders.csv", "coffee_orders.csv"):
    if os.path.exists(p):
        CSV_PATH = p
        break

if CSV_PATH:
    coffee_df = pd.read_csv(CSV_PATH)
else:
    coffee_df = pd.DataFrame([
        {"order_id":"O-5001","date":"2026-03-07","item":"Cappuccino","size":"S","category":"Espresso Drink","price":3.75,"payment":"Cash","store":"Downtown"},
        {"order_id":"O-5002","date":"2026-03-07","item":"Mocha","size":"S","category":"Espresso Drink","price":4.50,"payment":"Card","store":"Airport"},
        {"order_id":"O-5003","date":"2026-03-05","item":"Cappuccino","size":"L","category":"Espresso Drink","price":5.25,"payment":"Card","store":"Downtown"},
        {"order_id":"O-5004","date":"2026-03-05","item":"Croissant","size":"M","category":"Food","price":3.25,"payment":"App","store":"Uptown"},
        {"order_id":"O-5005","date":"2026-03-06","item":"Latte","size":"L","category":"Espresso Drink","price":5.50,"payment":"App","store":"Downtown"},
    ])

# --- Build run_sql(sql) -> DataFrame ---
if BACKEND == "duckdb":
    try:
        import duckdb
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
        import duckdb

    _con = duckdb.connect(database=":memory:")          # a private, in-memory warehouse
    _con.register("coffee_src", coffee_df)               # expose the DataFrame to SQL
    _con.execute("CREATE OR REPLACE TABLE coffee_orders AS SELECT * FROM coffee_src")

    def run_sql(sql: str) -> pd.DataFrame:
        """Run SQL and return the result as a pandas DataFrame."""
        return _con.execute(sql).df()

else:  # BACKEND == "snowflake"
    import snowflake.connector
    _con = snowflake.connector.connect(
        account=os.environ["SNOWFLAKE_ACCOUNT"],
        user=os.environ["SNOWFLAKE_USER"],
        password=os.environ["SNOWFLAKE_PASSWORD"],
        warehouse=os.environ.get("SNOWFLAKE_WAREHOUSE"),
        database=os.environ.get("SNOWFLAKE_DATABASE"),
        schema=os.environ.get("SNOWFLAKE_SCHEMA"),
    )
    # Make sure a coffee_orders table exists to query (idempotent load).
    _con.cursor().execute("CREATE TABLE IF NOT EXISTS coffee_orders (order_id STRING, date STRING, item STRING, size STRING, category STRING, price FLOAT, payment STRING, store STRING)")

    def run_sql(sql: str) -> pd.DataFrame:
        """Run SQL and return the result as a pandas DataFrame."""
        cur = _con.cursor()
        cur.execute(sql)
        return cur.fetch_pandas_all()

print(f"✅ Ready. Backend = {BACKEND.upper()}   (rows loaded: {len(coffee_df)})")
print("Try it:")
run_sql("SELECT order_id, item, price FROM coffee_orders LIMIT 3")

### 1 · `SELECT` — pick the columns you want  →  like pandas `df[[...]]`

Every query starts with **`SELECT`** (which columns) and **`FROM`** (which table). This is exactly last week's `df[["item", "price"]]` — you're just asking the *warehouse* to do it.

In [ ]:
run_sql("""
    SELECT item, price
    FROM coffee_orders
""").head()

In [ ]:
# SELECT *  means "every column".  (Whitespace and line breaks don't matter to SQL.)
run_sql("SELECT * FROM coffee_orders").head()

**What just happened:** `SELECT item, price FROM coffee_orders` returned just those two columns; `SELECT *` returned all of them. The result comes back as a **pandas DataFrame**, so `.head()` works like always. SQL keywords (`SELECT`, `FROM`) are conventionally UPPERCASE, but SQL doesn't care about case *or* line breaks — only the order of the clauses.

### 2 · `LIMIT` — only bring back a few rows  →  like `df.head()`

Against a 50-million-row table you **never** want every row while you're exploring. **`LIMIT n`** caps the result at `n` rows — the SQL twin of `df.head(n)`, and a good habit that keeps a stray query from pulling the whole warehouse.

In [ ]:
run_sql("""
    SELECT order_id, item, price
    FROM coffee_orders
    LIMIT 5
""")

### 3 · `WHERE` — keep only the rows you want  →  like `df[df[...] > 5]`

**`WHERE`** filters rows with a condition. It's pandas boolean filtering, in English word order.

⚠️ Two things trip everyone up at first: **text values go in single quotes** (`'Downtown'`), and **equality is a single `=`** (not `==` like Python).

In [ ]:
# premium orders  (pandas: df[df["price"] > 5])
run_sql("""
    SELECT order_id, item, price
    FROM coffee_orders
    WHERE price > 5
""")

In [ ]:
# combine conditions with AND / OR  — Downtown AND premium
run_sql("""
    SELECT order_id, item, price, store
    FROM coffee_orders
    WHERE store = 'Downtown' AND price > 5
""")

**What just happened:** `WHERE price > 5` kept only the pricey orders; `AND` chained a second condition. In pandas that second one was `df[(df["store"]=="Downtown") & (df["price"]>5)]` — here it's plain `AND`, no parentheses gymnastics.

### 4 · `ORDER BY` — sort the result  →  like `df.sort_values(...)`

**`ORDER BY column`** sorts ascending; add **`DESC`** for descending (biggest first). Same idea as `sort_values("price", ascending=False)`.

In [ ]:
# priciest orders first
run_sql("""
    SELECT order_id, item, price
    FROM coffee_orders
    ORDER BY price DESC
""").head()

### 5 · Put it together — the clauses always go in this order

`SELECT` → `FROM` → `WHERE` → `ORDER BY` → `LIMIT`. That order is fixed. Read the query below like a sentence: *"Show the order, item, and price of Downtown orders over \$5, priciest first, top 3."*

In [ ]:
run_sql("""
    SELECT order_id, item, price
    FROM coffee_orders
    WHERE store = 'Downtown' AND price > 5
    ORDER BY price DESC
    LIMIT 3
""")

> **🔗 The SQL ⇄ pandas cheat-sheet.** Everything today has a one-to-one twin with Week 2:
>
> | Goal | SQL | pandas |
> |---|---|---|
> | pick columns | `SELECT item, price` | `df[["item","price"]]` |
> | all columns | `SELECT *` | `df` |
> | filter rows | `WHERE price > 5` | `df[df["price"] > 5]` |
> | two conditions | `WHERE a AND b` | `df[(a) & (b)]` |
> | sort | `ORDER BY price DESC` | `df.sort_values("price", ascending=False)` |
> | first n rows | `LIMIT 5` | `df.head(5)` |
>
> You already *think* in these moves. SQL is just the words the warehouse understands.

> **`Go Deeper 🔧` — `DISTINCT`, aliases (`AS`), and comments.**
> - **`SELECT DISTINCT store`** returns each unique value once (pandas `df["store"].unique()`).
> - **`AS`** renames a column in the output: `SELECT price AS dollars`.
> - **`--`** starts a comment to the end of the line; `/* ... */` comments a block.

In [ ]:
run_sql("""
    SELECT DISTINCT store AS location   -- one row per store, column renamed
    FROM coffee_orders
    ORDER BY location
""")

> **`Common pitfalls ⚠️`**
>
> - **Strings need single quotes:** `WHERE item = 'Latte'`, not `"Latte"` and not `Latte`.
> - **Equality is one `=`** in SQL (`WHERE size = 'L'`), unlike Python's `==`.
> - **Clause order is fixed:** `WHERE` always comes before `ORDER BY`, which comes before `LIMIT`.
> - **`LIMIT` is your friend** while exploring a big warehouse table — add it by reflex.

### ✍️ Your turn

Fill in the SQL. Each `run_sql("""...""")` returns a DataFrame, so you'll see the result immediately.

In [ ]:
# TODO 1: select order_id, item, price for orders paid with 'App'
run_sql("""
    -- your SQL here
    SELECT order_id, item, price
    FROM coffee_orders
""")

# TODO 2: of those App orders, show only the ones over $4, priciest first

# TODO 3: list the DISTINCT payment methods in the table


<details><summary>✅ Show solution</summary>

```python
# 1
run_sql("""
    SELECT order_id, item, price
    FROM coffee_orders
    WHERE payment = 'App'
""")

# 2
run_sql("""
    SELECT order_id, item, price
    FROM coffee_orders
    WHERE payment = 'App' AND price > 4
    ORDER BY price DESC
""")

# 3
run_sql("""
    SELECT DISTINCT payment
    FROM coffee_orders
""")
```
</details>

### 🚀 Build the artifact — a reusable "top premium orders" query

A real, repeatable report in one function: pass a store, get its priciest orders back **as a DataFrame** you can chart, export, or hand to Claude. Notice the pattern — **SQL does the filtering/sorting in the warehouse; Python takes it from there.** That hand-off is the whole theme of Day 4.

In [ ]:
def top_premium(store: str, min_price: float = 5.0, n: int = 5) -> pd.DataFrame:
    """Priciest orders at one store, straight from the warehouse into pandas."""
    return run_sql(f"""
        SELECT order_id, item, price, store
        FROM coffee_orders
        WHERE store = '{store}' AND price >= {min_price}
        ORDER BY price DESC
        LIMIT {n}
    """)

report = top_premium("Downtown")
print(f"{len(report)} premium orders at Downtown — total ${report['price'].sum():.2f}")
report

> **🔗 Your world — from coffee to matters.** This *is* a matters query. Next week the table is `matters`, and the same four verbs answer real questions:
>
> ```sql
> SELECT matter_id, client, amount_billed        -- SELECT: the columns
> FROM matters
> WHERE status = 'Active' AND amount_billed > 50000   -- WHERE: big active matters
> ORDER BY amount_billed DESC                     -- ORDER BY: biggest first
> LIMIT 10;                                        -- LIMIT: the top ten
> ```
>
> Same shape, real stakes: *the ten largest active matters, biggest first.* That's the query that feeds a partner dashboard — and, later, *Matter Intelligence*.

### 📝 Recap — what you shipped

- A **data warehouse** holds data too big for your laptop; you send it **SQL** and get back just the answer.
- One helper — **`run_sql(...)`** — runs the *same SQL* on **Snowflake** (with credentials) or **DuckDB** (offline). You wrote real Snowflake SQL today without an account.
- The four core verbs: **`SELECT`** (columns), **`WHERE`** (filter), **`ORDER BY`** (sort), **`LIMIT`** (cap) — each a twin of a pandas move you already knew.
- Results come back as a **pandas DataFrame**, so the Python toolkit picks up where SQL leaves off.
- **Artifact:** `top_premium(store)` — a reusable warehouse query returning a ready-to-use DataFrame.

### 🧠 Check your understanding

1. Which SQL keyword is the twin of pandas `df[df["price"] > 5]`?
2. Why wrap text values in single quotes, and what's the SQL operator for "is equal to"?
3. In what order must `WHERE`, `ORDER BY`, and `LIMIT` appear?
4. This notebook ran without a Snowflake account. What actually executed your SQL, and why does that matter for learning?

<details><summary>✅ Answers</summary>

1. **`WHERE`** — it filters rows by a condition.
2. Text is a **string literal**, marked by single quotes (`'Latte'`); equality is a single **`=`** (not `==`).
3. **`WHERE` → `ORDER BY` → `LIMIT`** — that clause order is fixed.
4. **DuckDB**, a local engine whose SQL dialect is very close to Snowflake's. It lets you learn *real* warehouse SQL offline; the day credentials appear in `.env`, the identical queries run on Snowflake.
</details>

### ➡️ Next up — Week 6, Day 2: aggregations (`GROUP BY`, `COUNT` / `SUM` / `AVG`)

Today you picked, filtered, and sorted individual rows. Next we **summarize** them: *total revenue per category*, *average order value per store*, *how many orders each day* — the SQL twin of pandas `groupby`, and the verb that turns a table of rows into a report.

*Nothing to install — the same `run_sql` helper carries over.*

### 📖 Reference & glossary

| Term | Plain meaning | pandas twin |
|---|---|---|
| **Data warehouse** | a big shared database built for analytics (e.g. Snowflake) | — |
| **Table** | rows and named columns | a DataFrame |
| **`SELECT`** | choose columns | `df[["a","b"]]` |
| **`FROM`** | which table | `pd.read_csv(...)` source |
| **`WHERE`** | keep matching rows | `df[condition]` |
| **`ORDER BY ... DESC`** | sort (descending) | `df.sort_values(..., ascending=False)` |
| **`LIMIT n`** | first *n* rows | `df.head(n)` |
| **`DISTINCT`** | unique values only | `df[col].unique()` |
| **`AS`** | rename a column in output | `df.rename(...)` |
| **DuckDB** | fast local SQL engine used as our offline Snowflake stand-in | — |

**Commands used:** `SELECT`, `FROM`, `WHERE`, `AND`/`OR`, `ORDER BY`, `DESC`, `LIMIT`, `DISTINCT`, `AS`, `--` comments.

**Docs:** Snowflake SQL — https://docs.snowflake.com/en/sql-reference-commands · DuckDB SQL — https://duckdb.org/docs/sql/introduction · The models/config note and `.env` pattern live in the course `README.md`.

> *Not legal advice — these lessons teach technology. A lawyer reviews any AI or data output that will be relied upon.*